In [1]:
%pip install seqeval
%pip install tranformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=7813de70783276a8323801c8e8a6edd7cb3acb5af4aa0c95b56a8ff77d2f4b77
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
ERROR: Could not find a version that satisfies the requirement tranformers (from versions: none)
ERROR: No matching distribution found for tranformers


In [2]:
# Standard Library Imports
import os
import warnings
import json
from itertools import product

# Third-party Imports 
import numpy as np
import wandb

# PyTorch
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from torch.nn import CrossEntropyLoss

# Hugging Face / Datasets
from datasets import Dataset
from transformers import BertForTokenClassification, BertTokenizerFast

# Metrics & Evaluation
from sklearn.metrics import classification_report, accuracy_score, f1_score
from seqeval.metrics import classification_report as seqeval_classification_report

In [3]:
import warnings
import torch
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
class CustomDataset:
    def __init__(self, data):
        self.data = data
        self.train_data, self.val_data = self.data['train'], self.data['validation']

    def flatten(self, subset):
        flattened_data = []
        for item in subset:
            for j in range(len(item['input_ids'])):
                if len(item['input_ids'][j]) == 384:
                    flattened_data.append({
                        'input_ids': torch.tensor(item['input_ids'][j], dtype=torch.long),
                        'attention_mask': torch.tensor(item['attention_mask'][j], dtype=torch.long),
                        'labels': torch.tensor(item['labels'][j], dtype=torch.long),
                    })
        return flattened_data

    def create_data_loader(self, subset, batch_size = 5, shuffle=True, num_workers=0):
        flattened_subset = self.flatten(subset)
        loader_params = {
            'batch_size': batch_size,
            'shuffle': shuffle,
            'num_workers': num_workers
        }
        data_loader = DataLoader(flattened_subset, **loader_params)
        return data_loader
    
class Trainer:
    """
    Handles the fine-tuning of mBERT for NER on the ElCardioCC dataset.

    This class manages the training loop, validation, hyperparameter grid search, 
    and integration with Weights & Biases for experiment tracking.

    Example:
        >>> hyperparameters = {
        ...     'learning_rate': [8e-6],
        ...     'criterion': [CrossEntropyLoss],
        ...     'optimizer': [AdamW],
        ...     'max_grad_norm': [3],
        ...     'num_epochs': [5],
        ...     'weight_decay': [0.01]
        ... }
        ... wandb.login()
        >>> trainer = Trainer(dummy_data)
        >>> trainer.train(hyperparameters)
        >>> trainer.save_best_model("path/to/save", hyperparameters)
    """
    def __init__(self, data, device='cuda'):
      torch.manual_seed(42)
      np.random.seed(42)
      custom_dataset = CustomDataset(data)
      self.training_loader = custom_dataset.create_data_loader(custom_dataset.train_data)
      self.validation_loader = custom_dataset.create_data_loader(custom_dataset.val_data)
      self.device = device
      self.id2label = {0: 'O', 1: 'B-ENTITY',2: 'I-ENTITY'}
      self.label2id = {'O': 0, 'B-ENTITY': 1, 'I-ENTITY': 2}
      self.best_params = None

    def train_epoch(self, idx1 = 384):
        tr_loss, tr_f1_score = 0, 0
        nb_tr_steps = 0
        tr_preds, tr_labels = [], []
        self.model.train()

        for idx, batch in enumerate(self.training_loader):
            ids = batch['input_ids'].to(self.device, dtype=torch.long)
            mask = batch['attention_mask'].to(self.device, dtype=torch.long)
            targets = batch['labels'].to(self.device, dtype=torch.long)

            outputs = self.model(input_ids=ids, attention_mask=mask, labels=targets)
            tr_logits = outputs.logits
            first_logits = tr_logits[:, :idx1, :]
            first_targets = targets[:, :idx1]
            flattened_logits = first_logits.reshape(-1, self.model.num_labels)
            flattened_targets = first_targets.reshape(-1)
            flattened_predictions = torch.argmax(flattened_logits, axis=1)

            nb_tr_steps += 1

            active = mask[:, :idx1].reshape(-1) == 1
            logits = flattened_logits[active]
            targets = torch.masked_select(flattened_targets, active)
            predictions = torch.masked_select(flattened_predictions, active)

            loss = self.criterion(logits, targets)
            tr_loss += loss.item()

            tr_preds.extend(predictions.cpu())
            tr_labels.extend(targets.cpu())

            tmp_tr_f1_score = f1_score(targets.cpu().numpy(), predictions.cpu().numpy(), average = 'macro')
            tr_f1_score += tmp_tr_f1_score

            torch.nn.utils.clip_grad_norm_(
                parameters=self.model.parameters(), max_norm = self.max_grad_norm
            )

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        epoch_loss = tr_loss / nb_tr_steps
        tr_f1_score = tr_f1_score / nb_tr_steps
        f1 = f1_score(tr_labels, tr_preds, average='macro')
        wandb.log({"Epoch": self.epoch + 1, "Train Loss": epoch_loss, "Batch Train F1 ": tr_f1_score, 'Overall Train F1': f1})

        return epoch_loss, tr_f1_score

    def evaluate(self, loader, idx2):
        self.model.eval()
        eval_loss, eval_f1_score = 0, 0
        nb_eval_steps = 0
        eval_preds, eval_labels = [], []

        with torch.no_grad():
            for idx, batch in enumerate(loader):

                ids = batch['input_ids'].to(self.device, dtype = torch.long)
                mask = batch['attention_mask'].to(self.device, dtype = torch.long)
                targets = batch['labels'].to(self.device, dtype = torch.long)

                outputs = self.model(input_ids=ids, attention_mask=mask, labels=targets)
                eval_logits = outputs.logits

                first_logits = eval_logits[:, :idx2, :]
                first_targets = targets[:, :idx2]

                flattened_logits = first_logits.reshape(-1, self.model.num_labels)
                flattened_targets = first_targets.reshape(-1)
                flattened_predictions = torch.argmax(flattened_logits, axis=1)

                nb_eval_steps += 1

                active = mask[:, :idx2].reshape(-1) == 1
                logits = flattened_logits[active]
                targets = torch.masked_select(flattened_targets, active)
                predictions = torch.masked_select(flattened_predictions, active)

                loss = self.criterion(logits, targets)
                eval_loss += loss.item()

                eval_labels.extend(targets)
                eval_preds.extend(predictions)

                tmp_eval_f1_score = f1_score(targets.cpu().numpy(), predictions.cpu().numpy(), average = 'macro')
                eval_f1_score += tmp_eval_f1_score

        labels = [self.id2label[id.item()] for id in eval_labels]
        predictions = [self.id2label[id.item()] for id in eval_preds]
        f1 = f1_score(labels, predictions, average = 'macro')

        eval_loss = eval_loss / nb_eval_steps
        eval_f1_score = eval_f1_score / nb_eval_steps
        wandb.log({"Epoch": self.epoch + 1, "Eval Loss": eval_loss, 'Batch Eval F1': eval_f1_score,  'Overall Eval F1': f1})

        return eval_loss, eval_f1_score, labels, predictions, f1

    def train(self, hyperparameter_grid):
        self.best_model_state_dict = None
        self.best_cross_entropy = float('inf')
        self.best_f1 = 0
        self.best_params = None
        self.model = BertForTokenClassification.from_pretrained('bert-base-multilingual-cased',
                                                  num_labels=len(self.id2label),
                                                  id2label=self.id2label,
                                                  label2id=self.label2id)
        initial_model_state_dict = self.model.state_dict()
        torch.save(initial_model_state_dict, "initial_model_weights.pth")

        for i, params in enumerate(product(*hyperparameter_grid.values())):
            hyperparameters = dict(zip(hyperparameter_grid.keys(), params))
            print('-------------------------------------------------------------------------')
            print('Hyperparameters: ', hyperparameters)
            print('-------------------------------------------------------------------------')

            wandb.init(
                          project="ELCardioCC_NER_mBERT",
                          name= f"lr{hyperparameters['learning_rate']}",
                          config=hyperparameters
                      )

            self.model.load_state_dict(initial_model_state_dict)
            self.model.to(self.device)
            self.learning_rate = hyperparameters['learning_rate']
            self.weight_decay = hyperparameters['weight_decay']
            self.optimizer = hyperparameters['optimizer'](self.model.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
            self.criterion = hyperparameters['criterion']()
            self.max_grad_norm =  hyperparameters['max_grad_norm']
            self.num_epochs = hyperparameters['num_epochs']
            previous_eval_loss = float('inf')
            patience = 0

            trial_best_cross_entropy = float('inf')
            trial_best_f1 = 0
            trial_best_params = None

            for epoch in range(self.num_epochs):
                self.epoch = epoch
                epoch_loss, tr_f1_score = self.train_epoch(384)
                eval_loss, eval_f1_score, labels, predictions, f1 = self.evaluate(self.validation_loader, 384)
                print(f"Epoch {epoch + 1}/{self.num_epochs} => Train Loss: {epoch_loss:.4f}, Validation Loss: {eval_loss:.4f}")

                if eval_loss < trial_best_cross_entropy:
                    trial_best_model_state_dict = self.model.state_dict()
                    trial_best_params = hyperparameters
                    trial_best_params['best_epoch'] = epoch + 1
                    trial_best_cross_entropy = eval_loss
                    trial_best_f1 = f1

                if eval_loss > previous_eval_loss:
                    patience += 1
                    if patience >= 3:
                        print(f"Early stopping at epoch {epoch + 1}")
                        wandb.log({"early_stopping_epoch": epoch + 1})
                        break
                else:
                   patience = 0
                previous_eval_loss = eval_loss
            trial_best_model_info = {
                          'model_state_dict': trial_best_model_state_dict,
                          'hyperparameters': trial_best_params
                      }

            if trial_best_cross_entropy < self.best_cross_entropy:
                self.best_cross_entropy = trial_best_cross_entropy
                self.best_f1 = trial_best_f1
                self.best_params = trial_best_params
                self.best_model_state_dict = trial_best_model_state_dict
            wandb.finish()

        best_model_info = {
                              'model_state_dict': self.best_model_state_dict,
                              'hyperparameters': self.best_params
                          }

        print('-------------------------------------------------------------------------')
        print('Model Results:')
        print('-------------------------------------------------------------------------')
        print(f"Best Hyperparameters: {self.best_params}")
        print(f"Best Validation Loss: {self.best_cross_entropy}")

    def save_best_model(self, save_path, hyperparameter_grid):
        self.train(hyperparameter_grid)

        os.makedirs(save_path, exist_ok=True)
        self.model.load_state_dict(self.best_model_state_dict)
        self.model.to(self.device)

        self.model.save_pretrained(save_path)
        tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")
        tokenizer.save_pretrained(save_path)

        print(f"Best model saved to {save_path}")

        return self.model